In [ ]:
"""
Approximate Amplitude Encoding (AAE), Case 1, as used in

    Tezuka, Nakaji, Satoh, Yamamoto, "Grover search revisited; application to
    image pattern matching", arXiv:2108.10854 (Sec. II C, Sec. III B)
    (AAE itself: Nakaji et al., arXiv:2103.13211).

Scheme (Case 1: all amplitudes of the target are non-negative or non-positive)
-----------------------------------------------------------------------------
Train a PQC U(theta) so that  e^{i alpha} U(theta)|0>^n ~= |d> = sum_j d_j |j>.
This is equivalent to matching two probability distributions (Eqs. 20-21):

    |<j| U(theta) |0>|^2         = d_j^2
    |<j| H^{(x)n} U(theta) |0>|^2 = (d^H_j)^2 ,   d^H = H^{(x)n} d  (Walsh-Hadamard)

Cost (Eq. 22):  L = (L1 + L2) / 2,  with
    L1 = MMD^2( {|<j|U|0>|^2},      {d_j^2}     )
    L2 = MMD^2( {|<j|H^n U|0>|^2},  {(d^H_j)^2} )
and a Gaussian kernel  kappa(x, y) = exp(-64 (x - y)^2)  (Sec. III B).

Ansatz (Fig. 2): hardware-efficient ansatz; each layer is
    Ry(theta_r) on every qubit, followed by a CNOT ladder on adjacent qubits
(n-1 CNOTs per layer, e.g. 6 layers x 5 = 30 CNOTs for 6 qubits).

Training (Sec. III B): random initial angles, parameter-shift gradients of the
sampled distributions (q_theta^{+/-}, q_theta^{H +/-}), Adam optimizer with
learning rate 0.1 for the first 100 epochs and 0.01 afterwards.
"""

from __future__ import annotations

from typing import Callable, List, Optional, Sequence

import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def walsh_hadamard(d: np.ndarray) -> np.ndarray:
    """Normalized Walsh-Hadamard transform  d^H = H^{(x)n} d  in O(N log N)."""
    a = np.array(d, dtype=float, copy=True)
    n = a.size
    h = 1
    while h < n:
        a = a.reshape(-1, 2, h)
        a = np.stack([a[:, 0, :] + a[:, 1, :], a[:, 0, :] - a[:, 1, :]], axis=1)
        a = a.reshape(-1)
        h *= 2
    return a / np.sqrt(n)


def neqr_vector(pixels: Sequence[int], n_color_qubits: int) -> np.ndarray:
    """
    NEQR amplitude vector  (1/sqrt(N_P)) sum_j |g_j>_C |j>_P  (Sec. III A).
    Basis index = g_j * N_P + j  (colour register = high bits).
    Handy for building the target vector of the query / one database item.
    """
    n_p = len(pixels)
    vec = np.zeros((2 ** n_color_qubits) * n_p)
    for j, g in enumerate(pixels):
        vec[g * n_p + j] = 1.0 / np.sqrt(n_p)
    return vec


# --------------------------------------------------------------------------- #
# Approximate encoder
# --------------------------------------------------------------------------- #
class ApproximateEncoder:
    """
    Approximate amplitude encoder (AAE, Case 1) built on Qiskit.

    Parameters
    ----------
    n_qubits : number of qubits n (target vector length is 2**n).
    n_layers : number of HEA layers (paper: 6 for the 6-qubit database state
               A(theta_a), 3 for the 3-qubit query state B(theta_b)).
    kernel_gamma : Gaussian kernel exp(-gamma (x-y)^2); paper uses 64.
    normalize_index : if False (default, literal paper formula) the kernel is
               evaluated on integer indices j; if True on j / 2**n.
    n_shots : number of samples used to estimate each distribution
              (paper: 400 for A, 10000 for B). None -> exact probabilities.
    seed : RNG seed (initialisation and shot sampling).
    """

    def __init__(
        self,
        n_qubits: int,
        n_layers: int,
        kernel_gamma: float = 64.0,
        normalize_index: bool = False,
        n_shots: Optional[int] = None,
        seed: Optional[int] = None,
    ):
        self.n = n_qubits
        self.N = 2 ** n_qubits
        self.n_layers = n_layers
        self.n_params = n_qubits * n_layers
        self.gamma = kernel_gamma
        self.n_shots = n_shots
        self.rng = np.random.default_rng(seed)

        self._params = ParameterVector("theta", self.n_params)
        self.circuit = self._build_ansatz()                 # U(theta)
        self.circuit_H = self._build_ansatz(hadamard=True)  # H^{(x)n} U(theta)

        # Gaussian kernel matrix on integer indices j, k (Sec. II C / III B)
        idx = np.arange(self.N, dtype=float) / (self.N if normalize_index else 1.0)
        self._K = np.exp(-self.gamma * (idx[:, None] - idx[None, :]) ** 2)

        self.theta: Optional[np.ndarray] = None
        self.target: Optional[np.ndarray] = None
        self.sign: float = 1.0
        self.loss_history: List[float] = []

    # ------------------------------------------------------------------ #
    # Circuit construction
    # ------------------------------------------------------------------ #
    def _build_ansatz(self, hadamard: bool = False) -> QuantumCircuit:
        """Hardware-efficient ansatz: [Ry on all qubits -> CNOT ladder] x L."""
        qc = QuantumCircuit(self.n)
        r = 0
        for _ in range(self.n_layers):
            for q in range(self.n):
                qc.ry(self._params[r], q)
                r += 1
            for q in range(self.n - 1):
                qc.cx(q, q + 1)
        if hadamard:
            qc.h(range(self.n))
        return qc

    def get_circuit(self, theta: Optional[np.ndarray] = None) -> QuantumCircuit:
        """Return U(theta) with the (trained) angles bound; usable as A or B."""
        theta = self.theta if theta is None else theta
        if theta is None:
            raise RuntimeError("Call fit() first or pass theta explicitly.")
        return self.circuit.assign_parameters(np.asarray(theta, dtype=float))

    def get_inverse_circuit(self, theta: Optional[np.ndarray] = None) -> QuantumCircuit:
        """Return U(theta)^dagger (e.g. B^dagger in the inversion test)."""
        return self.get_circuit(theta).inverse()

    # ------------------------------------------------------------------ #
    # Distributions produced by the circuit
    # ------------------------------------------------------------------ #
    def _probabilities(self, circuit: QuantumCircuit, theta: np.ndarray) -> np.ndarray:
        bound = circuit.assign_parameters(theta)
        return Statevector(bound).probabilities()

    def _estimate(self, circuit: QuantumCircuit, theta: np.ndarray) -> np.ndarray:
        """Exact probabilities, or a finite-sample estimate if n_shots is set."""
        p = self._probabilities(circuit, theta)
        if self.n_shots is None:
            return p
        p = np.clip(p, 0.0, None)
        p = p / p.sum()
        return self.rng.multinomial(self.n_shots, p) / self.n_shots

    # ------------------------------------------------------------------ #
    # MMD loss (Eq. 22) and parameter-shift gradient
    # ------------------------------------------------------------------ #
    def _mmd2(self, q: np.ndarray, p: np.ndarray) -> float:
        diff = q - p
        return float(diff @ self._K @ diff)

    def _mmd2_grad_coeff(self, q: np.ndarray, p: np.ndarray) -> np.ndarray:
        """dL_MMD/dq = 2 K (q - p)."""
        return 2.0 * self._K @ (q - p)

    def loss(self, theta: Optional[np.ndarray] = None) -> float:
        """L = (L1 + L2)/2 evaluated with the current sampling setting."""
        theta = self.theta if theta is None else theta
        q1 = self._estimate(self.circuit, theta)
        q2 = self._estimate(self.circuit_H, theta)
        return 0.5 * (self._mmd2(q1, self._p1) + self._mmd2(q2, self._p2))

    def _gradient(self, theta: np.ndarray) -> np.ndarray:
        """
        Parameter-shift rule for Ry:
            dq_j/dtheta_r = [q_j(theta_r + pi/2) - q_j(theta_r - pi/2)] / 2
        applied to both distributions (q^{+/-}_theta and q^{H +/-}_theta).
        """
        q1 = self._estimate(self.circuit, theta)
        q2 = self._estimate(self.circuit_H, theta)
        c1 = self._mmd2_grad_coeff(q1, self._p1)
        c2 = self._mmd2_grad_coeff(q2, self._p2)

        grad = np.zeros_like(theta)
        for r in range(self.n_params):
            tp, tm = theta.copy(), theta.copy()
            tp[r] += np.pi / 2
            tm[r] -= np.pi / 2
            dq1 = 0.5 * (self._estimate(self.circuit, tp) - self._estimate(self.circuit, tm))
            dq2 = 0.5 * (self._estimate(self.circuit_H, tp) - self._estimate(self.circuit_H, tm))
            grad[r] = 0.5 * (c1 @ dq1 + c2 @ dq2)
        return grad

    # ------------------------------------------------------------------ #
    # Training
    # ------------------------------------------------------------------ #
    def fit(
        self,
        target: Sequence[float],
        n_iter: int = 500,
        lr_first: float = 0.1,
        lr_after: float = 0.01,
        lr_switch_epoch: int = 100,
        betas: tuple = (0.9, 0.999),
        eps: float = 1e-8,
        theta0: Optional[np.ndarray] = None,
        n_restarts: int = 1,
        callback: Optional[Callable[[int, float, np.ndarray], None]] = None,
        verbose: bool = False,
    ) -> "ApproximateEncoder":
        """
        Train U(theta) so that e^{i alpha} U(theta)|0> ~= |target> (Case 1).

        n_restarts > 1 repeats training from fresh random angles and keeps the
        run with the lowest final loss (the loss landscape has local minima;
        the paper simply initialises randomly at each training).

        Paper settings: n_iter=500 for A(theta_a), 300 for B(theta_b);
        Adam with lr 0.1 for the first 100 epochs, then 0.01.
        """
        d = np.asarray(target, dtype=float).ravel()
        if d.size != self.N:
            raise ValueError(f"target must have length 2**n = {self.N}, got {d.size}.")
        norm = np.linalg.norm(d)
        if norm == 0:
            raise ValueError("target vector must be non-zero.")
        d = d / norm

        # Case 1 only: all amplitudes non-negative or all non-positive.
        if np.all(d >= 0):
            self.sign = 1.0
        elif np.all(d <= 0):
            self.sign = -1.0            # absorbed in the global phase e^{i alpha}
            d = -d
        else:
            raise ValueError(
                "Mixed-sign amplitudes (AAE Case 2) are not implemented; "
                "only Case 1 (all >= 0 or all <= 0) is supported."
            )
        self.target = d

        # Target distributions of Eqs. (20)-(21)
        self._p1 = d ** 2
        self._p2 = walsh_hadamard(d) ** 2

        best_theta, best_loss, best_hist = None, np.inf, []
        for k in range(n_restarts):
            init = theta0 if (theta0 is not None and k == 0) else None
            theta, hist = self._train(init, n_iter, lr_first, lr_after,
                                      lr_switch_epoch, betas, eps, callback, verbose)
            if hist[-1] < best_loss:
                best_theta, best_loss, best_hist = theta, hist[-1], hist
        self.theta = best_theta
        self.loss_history = best_hist
        return self

    def _train(self, theta0, n_iter, lr_first, lr_after, lr_switch_epoch,
               betas, eps, callback, verbose):
        theta = (
            self.rng.uniform(0.0, 2.0 * np.pi, self.n_params)
            if theta0 is None else np.asarray(theta0, dtype=float).copy()
        )

        # Adam
        m = np.zeros_like(theta)
        v = np.zeros_like(theta)
        b1, b2 = betas
        history = []

        for t in range(1, n_iter + 1):
            lr = lr_first if t <= lr_switch_epoch else lr_after
            g = self._gradient(theta)
            m = b1 * m + (1 - b1) * g
            v = b2 * v + (1 - b2) * g ** 2
            m_hat = m / (1 - b1 ** t)
            v_hat = v / (1 - b2 ** t)
            theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)

            loss = self.loss(theta)
            history.append(loss)
            if callback is not None:
                callback(t, loss, theta)
            if verbose and (t % 50 == 0 or t == 1):
                print(f"[AAE] iter {t:4d}  loss = {loss:.3e}")

        return theta, history

    # ------------------------------------------------------------------ #
    # Evaluation
    # ------------------------------------------------------------------ #
    def get_statevector(self) -> np.ndarray:
        """Amplitudes of U(theta)|0> (real, since Ry and CNOT are real)."""
        return np.real_if_close(Statevector(self.get_circuit()).data)

    def fidelity(self) -> float:
        """|<target| U(theta)|0>|^2, insensitive to the global phase."""
        if self.target is None:
            raise RuntimeError("Call fit() first.")
        psi = Statevector(self.get_circuit()).data
        return float(abs(np.vdot(self.sign * self.target, psi)) ** 2)

    def n_cnots(self) -> int:
        return self.n_layers * (self.n - 1)


# --------------------------------------------------------------------------- #
# Example: encode the 3-qubit query state (4-pixel binary image) as in Sec. III
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    # 4-pixel binary image "6h" = 0110 -> pixels g_j = [0, 1, 1, 0], n_C = 1, n_P = 2
    query_vec = neqr_vector([0, 1, 1, 0], n_color_qubits=1)

    # Paper: B(theta_b) = 3-layer HEA, 300 iterations, 10000 samples
    B = ApproximateEncoder(n_qubits=3, n_layers=3, n_shots=10000, seed=0)
    B.fit(query_vec, n_iter=300, n_restarts=5, verbose=True)
    print("fidelity :", B.fidelity())
    print("CNOTs    :", B.n_cnots())
    print(B.get_circuit().draw())